# SigLIP2 PPO Text-to-Drawing Agent

End-to-end seed experiment notebook.

This version can run either one seed for quick experiments or four controlled seeds for final reporting. After training, it automatically evaluates every selected seed, saves reward/similarity curves, and creates a combined zip archive.

## 1. Install missing dependencies


In [ ]:
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "transformers": "transformers",
    "accelerate": "accelerate",
    "safetensors": "safetensors",
}

missing = [pkg for module, pkg in REQUIRED_PACKAGES.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All required packages are already installed.")


## 2. Imports


In [ ]:
import gc
import glob
import html
import json
import math
import os
import random
import time
from collections import deque
from typing import Optional, Tuple

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from IPython.display import HTML, clear_output, display
from PIL import Image, ImageDraw
from torch.distributions import Normal
from transformers import AutoModel, AutoProcessor


## 3. Experiment configuration

This version supports two modes:

1. `RUN_MODE = "single"`: train and evaluate only `SINGLE_SEED`.
2. `RUN_MODE = "four_seed"`: train and evaluate the four fixed seeds `[42, 123, 469, 1234]`.

After training, the notebook automatically evaluates the selected seed folders, saves reward/similarity curves, and zips the selected outputs. No post-training flag switching is needed.

In [ ]:
# Prompt and run length.
PROMPT = "A painting of a sunflower"
N_EPISODES = 5000

# Save folder: works on Kaggle and Colab without editing.
BASE_SAVE_DIR = "/kaggle/working/outputs_siglip" if os.path.isdir("/kaggle/working") else "/content/outputs_siglip"

# Seed mode.
# Use "single" for quick experiments, or "four_seed" for the final controlled experiment.
RUN_MODE = "four_seed"          # "single" or "four_seed"
SINGLE_SEED = 469             # used only when RUN_MODE = "single"
FOUR_SEEDS = [42, 123, 469, 1234]

# Selected seeds are derived from RUN_MODE. Do not edit RUN_SEEDS directly.
if RUN_MODE == "single":
    RUN_SEEDS = [int(SINGLE_SEED)]
elif RUN_MODE == "four_seed":
    RUN_SEEDS = [int(s) for s in FOUR_SEEDS]
else:
    raise ValueError("RUN_MODE must be 'single' or 'four_seed'.")

USE_SEED_SUBFOLDERS = True

# Optional: set to a specific existing folder only if you want to evaluate/zip a previous run manually.
# Keep None for the normal end-to-end workflow.
EVAL_SAVE_DIR_OVERRIDE = None
ZIP_OUTPUTS = True

# Colab Drive handling. If True, the notebook will ask to mount Drive when renderer.pkl is not found locally.
MOUNT_DRIVE_IF_COLAB = True

# Renderer path:
# - Use "auto" to search common Kaggle/Colab locations for renderer.pkl.
# - Or set an exact path, e.g. "/kaggle/input/.../renderer.pkl" or "/content/drive/MyDrive/renderer.pkl".
RENDERER_PATH = "auto"

# Optional reference sketch path. Keep None for pure text guidance.
SKETCH_PATH = None

# Vision-language reward model.
VL_MODEL_ID = "google/siglip2-base-patch16-224"
DEVICE_NAME = "auto"          # "auto", "cuda", "cpu", or "mps"
VL_DTYPE_NAME = "float32"     # "float32", "float16", "bfloat16", or "auto"

# Live display / saving.
LIVE_DISPLAY = True
DISPLAY_EVERY = 25            # lower values slow notebooks down
DISPLAY_SCALE = 2
SAVE_VISUAL_EVERY = 100       # saves visual_epXXXXX.png checkpoints


## 4. Device, paths, display, and reproducibility


In [ ]:
def running_in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False


def get_device(device_name: str = "auto") -> torch.device:
    if device_name == "auto":
        if torch.cuda.is_available():
            return torch.device("cuda")
        if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            return torch.device("mps")
        return torch.device("cpu")
    return torch.device(device_name)


def parse_dtype(dtype_name: str, device: torch.device) -> torch.dtype:
    dtype_name = dtype_name.lower()
    if device.type == "cpu" and dtype_name in {"float16", "bfloat16"}:
        print("[WARN] Using float32 on CPU; half precision is not reliable for this notebook.")
        return torch.float32
    if dtype_name == "float32":
        return torch.float32
    if dtype_name == "float16":
        return torch.float16
    if dtype_name == "bfloat16":
        return torch.bfloat16
    if dtype_name == "auto":
        return torch.float16 if device.type == "cuda" else torch.float32
    raise ValueError(f"Unsupported dtype: {dtype_name}")


def _is_colab_drive_mounted() -> bool:
    return os.path.isdir("/content/drive/MyDrive")


def maybe_mount_colab_drive(path: str = "/content/drive") -> bool:
    """Mount Google Drive in Colab when needed.

    Returns True when Drive appears mounted after the call.
    """
    if not running_in_colab():
        return False
    if _is_colab_drive_mounted():
        return True
    try:
        from google.colab import drive
        print("[INFO] renderer.pkl was not found locally. Mounting Google Drive so the notebook can search MyDrive...")
        drive.mount("/content/drive")
    except Exception as exc:
        print(f"[WARN] Could not mount Google Drive automatically: {exc}")
    return _is_colab_drive_mounted()


def _renderer_search_candidates(include_drive: bool = True) -> list[str]:
    patterns = [
        "/kaggle/input/**/renderer.pkl",
        "/content/renderer.pkl",
        "/content/**/renderer.pkl",
        "./renderer.pkl",
        "renderer.pkl",
    ]
    if include_drive:
        patterns.extend([
            "/content/drive/MyDrive/renderer.pkl",
            "/content/drive/MyDrive/**/renderer.pkl",
        ])

    candidates: list[str] = []
    for pattern in patterns:
        try:
            candidates.extend(glob.glob(pattern, recursive=True))
        except Exception:
            pass
    return sorted({p for p in candidates if os.path.isfile(p)})


def resolve_renderer_path(renderer_path: str = "auto") -> str:
    if renderer_path != "auto":
        if renderer_path.startswith("/content/drive"):
            maybe_mount_colab_drive(renderer_path)
        if not os.path.isfile(renderer_path):
            raise FileNotFoundError(
                f"Renderer file not found: {renderer_path}\n"
                "If it is in Google Drive, mount Drive first or keep MOUNT_DRIVE_IF_COLAB = True."
            )
        return renderer_path

    # First search without forcing a Drive mount. This is fast for Kaggle and Colab uploaded files.
    candidates = _renderer_search_candidates(include_drive=_is_colab_drive_mounted())

    # If not found in Colab, ask/mount Drive, then search MyDrive.
    if not candidates and running_in_colab() and MOUNT_DRIVE_IF_COLAB:
        maybe_mount_colab_drive()
        candidates = _renderer_search_candidates(include_drive=True)

    if not candidates:
        raise FileNotFoundError(
            "Could not find renderer.pkl.\n"
            "Fix one of these:\n"
            "1. Upload renderer.pkl to the current Colab session files, then rerun.\n"
            "2. Put renderer.pkl somewhere in Google Drive and keep MOUNT_DRIVE_IF_COLAB = True.\n"
            "3. Set RENDERER_PATH explicitly, for example:\n"
            "   RENDERER_PATH = '/content/drive/MyDrive/path/to/renderer.pkl'\n"
            "   or RENDERER_PATH = '/kaggle/input/<dataset>/renderer.pkl'"
        )

    # Prefer shorter paths if multiple copies exist, usually the intended one.
    candidates = sorted(candidates, key=lambda p: (len(p), p))
    print("Resolved renderer path:", candidates[0])
    if len(candidates) > 1:
        print("[INFO] Other renderer candidates found:")
        for extra in candidates[1:5]:
            print("  ", extra)
    return candidates[0]


def resolve_run_seed(seed: Optional[int]) -> int:
    """Return an explicit seed. If seed is None, generate one and record it.

    This lets exploratory runs remain random while still being reproducible later:
    copy the printed/generated seed back into SINGLE_SEED.
    """
    if seed is None:
        return int.from_bytes(os.urandom(4), byteorder="big", signed=False)
    return int(seed)


def set_seed(seed: int) -> None:
    seed = int(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def save_dir_for_seed(base_save_dir: str, seed: int, use_subfolders: bool = True) -> str:
    if not use_subfolders:
        return base_save_dir
    return os.path.join(base_save_dir, f"seed_{int(seed)}")


def canvas_np_to_pil(canvas_np: np.ndarray, scale: int = 2) -> Image.Image:
    arr = np.clip(canvas_np, 0.0, 1.0)
    img = Image.fromarray((arr * 255).astype(np.uint8)).convert("RGB")
    if scale != 1:
        img = img.resize((img.width * scale, img.height * scale), Image.NEAREST)
    return img


def make_side_by_side(current_np: np.ndarray, best_np: Optional[np.ndarray], scale: int = 2) -> Image.Image:
    current_img = canvas_np_to_pil(current_np, scale=scale)
    best_img = canvas_np_to_pil(best_np, scale=scale) if best_np is not None else Image.new("RGB", current_img.size, "white")

    gap = 20
    label_h = 28
    w, h = current_img.size
    panel = Image.new("RGB", (2 * w + gap, h + label_h), "white")
    draw = ImageDraw.Draw(panel)
    draw.text((0, 4), "Current canvas", fill="black")
    draw.text((w + gap, 4), "Best canvas", fill="black")
    panel.paste(current_img, (0, label_h))
    panel.paste(best_img, (w + gap, label_h))
    return panel


def fake_imshow(
    current_canvas_np: np.ndarray,
    best_canvas_np: Optional[np.ndarray] = None,
    metrics: Optional[dict] = None,
    title: str = "Training progress",
    save_path: Optional[str] = None,
    scale: int = 2,
):
    panel = make_side_by_side(current_canvas_np, best_canvas_np, scale=scale)
    if save_path is not None:
        panel.save(save_path)

    metric_lines = []
    if metrics:
        for key, value in metrics.items():
            if isinstance(value, float):
                metric_lines.append(f"{key}: {value:.4f}")
            else:
                metric_lines.append(f"{key}: {value}")

    clear_output(wait=True)
    display(HTML(f"<h3>{html.escape(title)}</h3><pre>{html.escape(chr(10).join(metric_lines))}</pre>"))
    display(panel)


DEVICE = get_device(DEVICE_NAME)
VL_DTYPE = parse_dtype(VL_DTYPE_NAME, DEVICE)

# Do not call set_seed(...) here. The seed is selected and applied inside run_one_seed(...)
# immediately before training. This is necessary because RUN_MODE can execute one seed
# or several seeds in the same notebook.
SAVE_DIR = BASE_SAVE_DIR  # backwards-compatible alias; actual seed runs save under seed_<seed>/ folders

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Device:", DEVICE)
print("VL dtype:", VL_DTYPE)
print("Run mode:", RUN_MODE)
print("Base save dir:", BASE_SAVE_DIR)
print("Single seed:", SINGLE_SEED)
print("Four seeds:", FOUR_SEEDS)
print("Run seeds:", RUN_SEEDS)


## 5. Hyperparameters


In [ ]:
CANVAS_SIZE = 128
RENDERER_PARAMS = 10
STROKE_COLOR = 3
STROKE_PARAMS = RENDERER_PARAMS + STROKE_COLOR  # 13

STROKES_PER_STEP = 5
N_STROKE_BUNDLES = 5

CNN_DIM = 256
STEP_DIM = 1
HIDDEN_DIM = 512
ACTION_DIM = STROKES_PER_STEP * STROKE_PARAMS

# Stroke vector layout: (x0, y0, x1, y1, x2, y2, r0, t0, r1, t1, R, G, B)
_POS_SLICE = slice(0, 6)
_COLOR_SLICE = slice(10, 13)

# Actor initialization. The strongest empirical fix was reducing initial log-std to -2.
_BIAS_R = -3.0
_BIAS_T = 0.0
ACTOR_LOG_STD_INIT = -2.0

# Reward.
REWARD_SCALE = 150.0
REWARD_CLIP = 100.0
TEXT_SIM_WEIGHT = 1.0
SKETCH_SIM_WEIGHT = 0.0
DIV_BONUS_W = 1
STROKE_SIZE_PENALTY_W = 5.0
TERMINAL_ABS_W = 150.0
USE_SQUARED_TEXT_SIM = False

# PPO.
LR = 5e-4
LR_MIN = 2e-6
GAMMA = 0.99
LAMBDA_GAE = 0.98
CLIP_EPS = 0.15       # PPO clip epsilon; unrelated to OpenAI CLIP
ENTROPY_START = 0.015
ENTROPY_END = 0.002
VALUE_COEF = 0.5
PPO_EPOCHS = 10
MINI_BATCH = 64
UPDATE_EVERY = 10 * N_STROKE_BUNDLES

# Logging / early stopping.
LOG_EVERY = DISPLAY_EVERY
PATIENCE = 3000


## 6. Utility functions


In [ ]:
def _safe_torch_load(path: str, device: torch.device):
    try:
        return torch.load(path, map_location=device, weights_only=True)
    except TypeError:
        return torch.load(path, map_location=device)


def _pil_bilinear():
    return getattr(Image, "Resampling", Image).BILINEAR


def _l2_normalize(x: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    return x / x.norm(dim=-1, keepdim=True).clamp_min(eps)


def compute_state_dim(vl_embed_dim: int) -> int:
    return (3 * int(vl_embed_dim)) + CNN_DIM + STEP_DIM


def save_json(obj: dict, path: str) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


## 7. SigLIP / SigLIP2 encoder


In [ ]:
# Local duplicate to make this cell robust if Utility Functions was skipped accidentally.
def _l2_normalize(x: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    return x / x.norm(dim=-1, keepdim=True).clamp_min(eps)

class VisionLanguageEncoder:
    # Small adapter around Hugging Face SigLIP/SigLIP2.

    def __init__(self, model_id: str, device: torch.device, dtype: torch.dtype = torch.float32, text_max_length: int = 64):
        self.model_id = model_id
        self.device = device
        self.dtype = dtype
        self.text_max_length = text_max_length

        print(f"[INFO] Loading vision-language model: {model_id}")
        self.processor = AutoProcessor.from_pretrained(model_id)
        try:
            self.model = AutoModel.from_pretrained(model_id, dtype=dtype)
        except TypeError:
            self.model = AutoModel.from_pretrained(model_id, torch_dtype=dtype)

        self.model = self.model.to(device)
        self.model.eval()
        for p in self.model.parameters():
            p.requires_grad_(False)

        self.image_size = self._infer_image_size()
        self.image_mean, self.image_std = self._infer_image_norm()

        with torch.no_grad():
            dummy = self.encode_text("dummy prompt")
        self.embed_dim = int(dummy.shape[-1])
        print(f"[VL] embed_dim={self.embed_dim} | image_size={self.image_size} | dtype={self.dtype}")

    def _infer_image_size(self) -> Tuple[int, int]:
        image_processor = getattr(self.processor, "image_processor", None)
        size = getattr(image_processor, "size", None) or {}
        if isinstance(size, dict):
            if "height" in size and "width" in size:
                return int(size["height"]), int(size["width"])
            if "shortest_edge" in size:
                edge = int(size["shortest_edge"])
                return edge, edge
        if isinstance(size, int):
            return size, size
        return 224, 224

    def _infer_image_norm(self) -> Tuple[torch.Tensor, torch.Tensor]:
        image_processor = getattr(self.processor, "image_processor", None)
        mean = getattr(image_processor, "image_mean", [0.5, 0.5, 0.5])
        std = getattr(image_processor, "image_std", [0.5, 0.5, 0.5])
        mean_t = torch.tensor(mean, device=self.device, dtype=self.dtype).view(1, 3, 1, 1)
        std_t = torch.tensor(std, device=self.device, dtype=self.dtype).view(1, 3, 1, 1)
        return mean_t, std_t

    @staticmethod
    def _as_feature_tensor(output, *, prefer: str = "pooler_output") -> torch.Tensor:
        if torch.is_tensor(output):
            return output
        candidate_names = ["image_embeds", "text_embeds", prefer, "pooler_output", "last_hidden_state"]
        for name in candidate_names:
            value = getattr(output, name, None)
            if torch.is_tensor(value):
                return value[:, 0] if name == "last_hidden_state" and value.ndim == 3 else value
        if isinstance(output, dict):
            for name in candidate_names:
                value = output.get(name)
                if torch.is_tensor(value):
                    return value[:, 0] if name == "last_hidden_state" and value.ndim == 3 else value
        if isinstance(output, (tuple, list)):
            for value in output:
                if torch.is_tensor(value):
                    return value[:, 0] if value.ndim == 3 else value
        raise TypeError(f"Could not extract tensor features from {type(output)}")

    def _image_features(self, pixel_values: torch.Tensor) -> torch.Tensor:
        if hasattr(self.model, "get_image_features"):
            out = self.model.get_image_features(pixel_values=pixel_values)
            return self._as_feature_tensor(out)
        if hasattr(self.model, "vision_model"):
            out = self.model.vision_model(pixel_values=pixel_values)
            features = self._as_feature_tensor(out)
            proj = getattr(self.model, "visual_projection", None) or getattr(self.model, "vision_projection", None)
            return proj(features) if proj is not None else features
        raise AttributeError("Loaded model does not expose image feature extraction.")

    def _text_features(self, **text_inputs) -> torch.Tensor:
        if hasattr(self.model, "get_text_features"):
            out = self.model.get_text_features(**text_inputs)
            return self._as_feature_tensor(out)
        if hasattr(self.model, "text_model"):
            out = self.model.text_model(**text_inputs)
            features = self._as_feature_tensor(out)
            proj = getattr(self.model, "text_projection", None)
            return proj(features) if proj is not None else features
        raise AttributeError("Loaded model does not expose text feature extraction.")

    def encode_text(self, text: str) -> torch.Tensor:
        inputs = self.processor(
            text=[text],
            padding="max_length",
            max_length=self.text_max_length,
            truncation=True,
            return_tensors="pt",
        )
        allowed = {"input_ids", "attention_mask", "position_ids"}
        inputs = {k: v.to(self.device) for k, v in inputs.items() if k in allowed}
        emb = self._text_features(**inputs).float()
        return _l2_normalize(emb)

    def preprocess_canvases(self, canvases: torch.Tensor) -> torch.Tensor:
        target_h, target_w = self.image_size
        imgs = F.interpolate(canvases.float(), size=(target_h, target_w), mode="bilinear", align_corners=False)
        imgs = imgs.to(device=self.device, dtype=self.dtype)
        return (imgs - self.image_mean) / self.image_std

    def encode_batch(self, canvases: torch.Tensor) -> torch.Tensor:
        pixel_values = self.preprocess_canvases(canvases)
        emb = self._image_features(pixel_values=pixel_values).float()
        return _l2_normalize(emb)

    def encode_image(self, img_tensor: torch.Tensor) -> torch.Tensor:
        return self.encode_batch(img_tensor.unsqueeze(0))


## 8. Frozen neural renderer


In [ ]:
class FCN(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(10, 512)
        self.fc2 = nn.Linear(512, 1024)
        self.fc3 = nn.Linear(1024, 2048)
        self.fc4 = nn.Linear(2048, 4096)
        self.conv1 = nn.Conv2d(16, 32, 3, 1, 1)
        self.conv2 = nn.Conv2d(32, 32, 3, 1, 1)
        self.conv3 = nn.Conv2d(8, 16, 3, 1, 1)
        self.conv4 = nn.Conv2d(16, 16, 3, 1, 1)
        self.conv5 = nn.Conv2d(4, 8, 3, 1, 1)
        self.conv6 = nn.Conv2d(8, 4, 3, 1, 1)
        self.pixel_shuffle = nn.PixelShuffle(2)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = F.relu(self.fc4(x))
        x = x.view(-1, 16, 16, 16)
        x = F.relu(self.conv1(x))
        x = self.pixel_shuffle(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = self.pixel_shuffle(self.conv4(x))
        x = F.relu(self.conv5(x))
        x = self.pixel_shuffle(self.conv6(x))
        x = torch.sigmoid(x)
        return 1 - x.view(-1, 128, 128)


class NeuralRenderer(nn.Module):
    def __init__(self, path: str, device: torch.device):
        super().__init__()
        self.net = FCN()
        state_dict = _safe_torch_load(path, device)
        self.net.load_state_dict(state_dict)
        self.net = self.net.to(device)
        self.net.eval()
        for p in self.net.parameters():
            p.requires_grad_(False)
        with torch.no_grad():
            dummy = torch.zeros(1, RENDERER_PARAMS, device=device)
            out = self.forward(dummy)
        assert out.dim() == 3, f"Unexpected renderer output shape: {out.shape}"
        self.out_h, self.out_w = out.shape[1], out.shape[2]
        print(f"[Renderer] {path} | alpha=({self.out_h}x{self.out_w})")

    def forward(self, params: torch.Tensor) -> torch.Tensor:
        out = self.net(params)
        if out.dim() == 4:
            out = out[:, 0]
        out = 1.0 - out
        return out.clamp(0.0, 1.0)


## 9. Canvas rendering operations


In [ ]:
def _composite(canvas: torch.Tensor, alpha: torch.Tensor, color: torch.Tensor) -> torch.Tensor:
    fore = color.view(3, 1, 1).expand_as(canvas)
    return (canvas * (1.0 - alpha.unsqueeze(0)) + fore * alpha.unsqueeze(0)).clamp(0.0, 1.0)


def _resize_alpha(alpha: torch.Tensor, H: int, W: int) -> torch.Tensor:
    if alpha.shape[-2:] == torch.Size([H, W]):
        return alpha
    return F.interpolate(alpha.unsqueeze(1), size=(H, W), mode="bilinear", align_corners=False).squeeze(1)


def render_bundle(canvas: torch.Tensor, action_01: torch.Tensor, renderer: NeuralRenderer) -> Tuple[torch.Tensor, float]:
    H, W = canvas.shape[1], canvas.shape[2]
    bundle = action_01.view(STROKES_PER_STEP, STROKE_PARAMS)
    total_alpha = 0.0

    for i in range(STROKES_PER_STEP):
        geo = bundle[i, :RENDERER_PARAMS]
        color = bundle[i, RENDERER_PARAMS:]
        alpha = renderer(geo.unsqueeze(0)).squeeze(0)
        alpha = _resize_alpha(alpha.unsqueeze(0), H, W).squeeze(0)
        total_alpha += alpha.detach().mean().item()
        canvas = _composite(canvas, alpha, color)

    return canvas, total_alpha / STROKES_PER_STEP


## 10. Optional sketch helper


In [ ]:
def load_sketch_embed(sketch_path: Optional[str], vl: VisionLanguageEncoder, device: torch.device) -> torch.Tensor:
    if sketch_path is None or not os.path.isfile(sketch_path):
        print("[Sketch] No sketch provided; sketch embedding is zero.")
        return torch.zeros(1, vl.embed_dim, device=device)
    img = Image.open(sketch_path).convert("RGB").resize((CANVAS_SIZE, CANVAS_SIZE), _pil_bilinear())
    img_t = torch.tensor(np.array(img, dtype=np.float32) / 255.0, device=device).permute(2, 0, 1)
    with torch.no_grad():
        emb = vl.encode_image(img_t)
    print(f"[Sketch] Encoded {sketch_path} -> {tuple(emb.shape)}")
    return emb


## 11. Canvas CNN encoder


In [ ]:
class CanvasCNN(nn.Module):
    def __init__(self, out_dim: int = CNN_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 5, stride=2, padding=2),
            nn.GroupNorm(8, 32),
            nn.GELU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.GroupNorm(8, 64),
            nn.GELU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1),
            nn.GroupNorm(16, 128),
            nn.GELU(),
            nn.Conv2d(128, 256, 3, stride=2, padding=1),
            nn.GroupNorm(32, 256),
            nn.GELU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
        )
        self.proj = nn.Linear(256, out_dim)

    def forward(self, canvas: torch.Tensor) -> torch.Tensor:
        return self.proj(self.net(canvas))


## 12. Actor-critic policy


In [ ]:
def _make_actor_bias() -> torch.Tensor:
    s = torch.zeros(STROKE_PARAMS)
    s[6] = _BIAS_R  # r0
    s[8] = _BIAS_R  # r1
    s[7] = _BIAS_T  # t0
    s[9] = _BIAS_T  # t1
    return s.repeat(STROKES_PER_STEP)


class ActorCritic(nn.Module):
    def __init__(self, vl_embed_dim: int):
        super().__init__()
        self.vl_embed_dim = int(vl_embed_dim)
        self.state_dim = compute_state_dim(self.vl_embed_dim)

        self.canvas_stream = nn.Sequential(
            nn.Linear(self.vl_embed_dim, HIDDEN_DIM),
            nn.LayerNorm(HIDDEN_DIM),
            nn.GELU(),
            nn.Linear(HIDDEN_DIM, HIDDEN_DIM // 2),
            nn.GELU(),
        )
        self.text_stream = nn.Sequential(
            nn.Linear(self.vl_embed_dim, HIDDEN_DIM // 2),
            nn.LayerNorm(HIDDEN_DIM // 2),
            nn.GELU(),
            nn.Linear(HIDDEN_DIM // 2, HIDDEN_DIM // 2),
            nn.GELU(),
        )
        self.canvas_cnn = CanvasCNN(out_dim=CNN_DIM)
        self.sketch_proj = nn.Sequential(nn.Linear(self.vl_embed_dim, 128), nn.GELU())
        self.step_proj = nn.Sequential(nn.Linear(STEP_DIM, 32), nn.GELU())

        fused_dim = HIDDEN_DIM // 2 + HIDDEN_DIM // 2 + CNN_DIM + 128 + 32
        self.trunk = nn.Sequential(
            nn.Linear(fused_dim, HIDDEN_DIM),
            nn.LayerNorm(HIDDEN_DIM),
            nn.GELU(),
            nn.Linear(HIDDEN_DIM, HIDDEN_DIM),
            nn.GELU(),
        )
        self.actor_mean = nn.Linear(HIDDEN_DIM, ACTION_DIM)
        self.actor_log_std = nn.Parameter(torch.full((ACTION_DIM,), ACTOR_LOG_STD_INIT))
        self.critic = nn.Sequential(nn.Linear(HIDDEN_DIM, 256), nn.GELU(), nn.Linear(256, 1))

        nn.init.orthogonal_(self.actor_mean.weight, gain=0.01)
        with torch.no_grad():
            self.actor_mean.bias.copy_(_make_actor_bias())

    def _unpack_state(self, state: torch.Tensor, canvas: torch.Tensor):
        e = self.vl_embed_dim
        clip_canvas = state[:, :e]
        clip_text = state[:, e:2 * e]
        clip_sketch = state[:, 2 * e + CNN_DIM:3 * e + CNN_DIM]
        step_norm = state[:, 3 * e + CNN_DIM:3 * e + CNN_DIM + 1]
        return clip_canvas, clip_text, canvas, clip_sketch, step_norm

    def _encode(self, clip_canvas, clip_text, canvas_px, clip_sketch, step_norm):
        c = self.canvas_stream(clip_canvas)
        t = self.text_stream(clip_text)
        v = self.canvas_cnn(canvas_px)
        sk = self.sketch_proj(clip_sketch)
        st = self.step_proj(step_norm)
        return self.trunk(torch.cat([c, t, v, sk, st], dim=-1))

    def forward(self, state: torch.Tensor, canvas: torch.Tensor):
        clip_canvas, clip_text, canvas_px, clip_sketch, step_norm = self._unpack_state(state, canvas)
        h = self._encode(clip_canvas, clip_text, canvas_px, clip_sketch, step_norm)
        mean = self.actor_mean(h)
        std = self.actor_log_std.exp().clamp(1e-4, 1.5).expand_as(mean)
        val = self.critic(h).squeeze(-1)
        return mean, std, val

    @torch.no_grad()
    def act(self, state: torch.Tensor, canvas: torch.Tensor, deterministic: bool = False):
        mean, std, val = self.forward(state.unsqueeze(0), canvas.unsqueeze(0))
        dist = Normal(mean, std)
        raw = mean if deterministic else dist.rsample()
        log_p = dist.log_prob(raw).sum(-1)
        return raw.squeeze(0).cpu().numpy(), log_p.squeeze(0), val.squeeze(0)

    def evaluate_actions(self, states: torch.Tensor, canvases: torch.Tensor, actions: torch.Tensor):
        mean, std, val = self.forward(states, canvases)
        dist = Normal(mean, std)
        log_p = dist.log_prob(actions).sum(-1)
        entropy = dist.entropy().sum(-1)
        return log_p, val, entropy


## 13. Rollout buffer


In [ ]:
class RolloutBuffer:
    def __init__(self):
        self.clear()

    def push(self, state, canvas_before_cpu, act, rew, lp, val, done):
        self._states.append(state.detach().cpu())
        self._canvases.append(canvas_before_cpu)
        self._acts.append(torch.tensor(act, dtype=torch.float32))
        self._rews.append(float(rew))
        self._lps.append(lp.detach().cpu())
        self._vals.append(val.detach().cpu())
        self._dones.append(bool(done))

    def __len__(self):
        return len(self._rews)

    def clear(self):
        self._states = []
        self._canvases = []
        self._acts = []
        self._rews = []
        self._lps = []
        self._vals = []
        self._dones = []


## 14. Environment


In [ ]:
class PaintEnv:
    def __init__(self, text_embed: torch.Tensor, sketch_embed: torch.Tensor, vl: VisionLanguageEncoder, renderer: NeuralRenderer, device: torch.device):
        self.text_embed = text_embed
        self.sketch_embed = sketch_embed
        self.vl = vl
        self.renderer = renderer
        self.device = device
        self.canvas = None
        self.episode_colors = []
        self.step_count = 0
        self.prev_sim = 0.0
        self.reset()

    def _encode_canvas(self):
        with torch.no_grad():
            return self.vl.encode_image(self.canvas)

    def _diversity_bonus(self) -> float:
        if len(self.episode_colors) < 2:
            return 0.0
        cols = np.array(self.episode_colors, dtype=np.float32)
        diffs = cols[:, None, :] - cols[None, :, :]
        n = len(cols)
        return float(np.linalg.norm(diffs, axis=-1)[np.triu_indices(n, k=1)].mean() / 1.732)

    def _similarity_score(self, emb: torch.Tensor) -> Tuple[float, float, float]:
        text_sim = (emb * self.text_embed).sum().item()
        text_component = text_sim ** 2 if USE_SQUARED_TEXT_SIM else text_sim
        if self.sketch_embed.abs().sum() > 1e-6:
            sketch_sim = (emb * self.sketch_embed).sum().item()
            reward_sim = TEXT_SIM_WEIGHT * text_component + SKETCH_SIM_WEIGHT * sketch_sim
        else:
            sketch_sim = 0.0
            reward_sim = text_component
        return float(reward_sim), float(text_sim), float(sketch_sim)

    def _build_state(self, canvas_embed: torch.Tensor) -> torch.Tensor:
        step_norm = torch.tensor([self.step_count / N_STROKE_BUNDLES], dtype=torch.float32, device=self.device)
        return torch.cat([
            canvas_embed.squeeze(0),
            self.text_embed.squeeze(0),
            torch.zeros(CNN_DIM, device=self.device),
            self.sketch_embed.squeeze(0),
            step_norm,
        ], dim=0)

    def reset(self):
        self.canvas = torch.ones(3, CANVAS_SIZE, CANVAS_SIZE, dtype=torch.float32, device=self.device)
        self.step_count = 0
        self.episode_colors = []
        emb = self._encode_canvas()
        self.prev_sim, _, _ = self._similarity_score(emb)
        return self._build_state(emb), self.canvas.detach()

    def get_canvas_snapshot(self):
        return self.canvas.detach().cpu().half()

    def step(self, action: np.ndarray):
        action_t = torch.tensor(action, dtype=torch.float32, device=self.device)
        action_01 = torch.sigmoid(action_t)

        with torch.no_grad():
            new_canvas, mean_alpha = render_bundle(self.canvas, action_01, self.renderer)
        self.canvas = new_canvas.detach()

        bundle = action_01.view(STROKES_PER_STEP, STROKE_PARAMS).detach().cpu()
        for i in range(STROKES_PER_STEP):
            self.episode_colors.append(tuple(bundle[i, RENDERER_PARAMS:].tolist()))

        emb = self._encode_canvas()
        new_sim, text_sim, sketch_sim = self._similarity_score(emb)
        delta_sim = float(np.clip((new_sim - self.prev_sim) * REWARD_SCALE, -REWARD_CLIP, REWARD_CLIP))
        self.prev_sim = new_sim

        stroke_penalty = -STROKE_SIZE_PENALTY_W * mean_alpha
        self.step_count += 1
        done = self.step_count >= N_STROKE_BUNDLES
        div_bonus = DIV_BONUS_W * self._diversity_bonus() if done else 0.0
        terminal_bonus = TERMINAL_ABS_W * text_sim if done else 0.0
        reward = delta_sim + stroke_penalty + div_bonus + terminal_bonus

        coverage = float((self.canvas.detach() < 0.95).any(dim=0).float().mean())
        state = self._build_state(emb)
        return state, self.canvas.detach(), reward, done, {
            "vl_sim": new_sim,
            "text_sim": text_sim,
            "sketch_sim": sketch_sim,
            "coverage": coverage,
            "mean_alpha": mean_alpha,
            "delta_sim_reward": delta_sim,
            "stroke_penalty": stroke_penalty,
            "div_bonus": div_bonus,
            "terminal_bonus": terminal_bonus,
            "reward": reward,
        }


## 15. GAE


In [ ]:
def compute_gae(rewards, values, dones, last_value):
    device = last_value.device
    dtype = last_value.dtype
    rewards = torch.tensor(rewards, dtype=dtype, device=device)
    dones = torch.tensor(dones, dtype=dtype, device=device)
    values = torch.stack([v.to(device=device, dtype=dtype) for v in values])
    n = len(rewards)
    advs = torch.zeros(n, dtype=dtype, device=device)
    gae = torch.zeros(1, dtype=dtype, device=device)
    prev = last_value
    for t in reversed(range(n)):
        mask = 1.0 - dones[t]
        delta = rewards[t] + GAMMA * prev * mask - values[t]
        gae = delta + GAMMA * LAMBDA_GAE * mask * gae
        advs[t] = gae
        prev = values[t]
    return advs, advs + values


## 16. PPO update


In [ ]:
def ppo_update(policy, optimizer, buffer, last_val, device, entropy_coef):
    states_t = torch.stack(buffer._states).to(device)
    acts_t = torch.stack(buffer._acts).to(device)
    old_lp_t = torch.stack(buffer._lps).to(device)
    canvases_cpu = buffer._canvases

    advs, rets = compute_gae(buffer._rews, buffer._vals, buffer._dones, last_val)
    advs_n = ((advs - advs.mean()) / (advs.std() + 1e-8)).to(device)
    rets = rets.to(device)
    T = len(buffer._rews)

    losses = []
    for _ in range(PPO_EPOCHS):
        idx = torch.randperm(T)
        for start in range(0, T, MINI_BATCH):
            mb = idx[start:start + MINI_BATCH]
            if len(mb) < 4:
                continue
            mb_canvases = torch.stack([
                canvases_cpu[i.item()].to(device=device, dtype=torch.float32) for i in mb
            ])
            new_lp, new_val, entropy = policy.evaluate_actions(states_t[mb], mb_canvases, acts_t[mb])
            ratio = (new_lp - old_lp_t[mb]).exp()
            surr1 = ratio * advs_n[mb]
            surr2 = ratio.clamp(1 - CLIP_EPS, 1 + CLIP_EPS) * advs_n[mb]
            actor_loss = -torch.min(surr1, surr2).mean()
            critic_loss = F.mse_loss(new_val, rets[mb])
            entropy_loss = -entropy.mean()
            loss = actor_loss + VALUE_COEF * critic_loss + entropy_coef * entropy_loss

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(policy.parameters(), max_norm=0.5)
            optimizer.step()
            losses.append(float(loss.detach().cpu()))
    return float(np.mean(losses)) if losses else 0.0


## 17. Plotting and saving helpers


In [ ]:
def _save_canvas(canvas_np: np.ndarray, path: str):
    Image.fromarray((np.clip(canvas_np, 0.0, 1.0) * 255).astype(np.uint8)).save(path)


def _plot_curves(sims, covs, alphas, rewards, prompt: str, save_dir: str):
    """Save a compact training curve with only similarity and reward."""
    if len(sims) == 0:
        return

    win = min(50, len(sims))

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 6), sharex=True)

    ax1.plot(sims, alpha=0.35, linewidth=0.8, label="VL similarity")
    if len(sims) >= win:
        smoothed = np.convolve(sims, np.ones(win) / win, mode="valid")
        ax1.plot(range(win - 1, len(sims)), smoothed, linewidth=2, label=f"{win}-episode average")
    ax1.set_ylabel("Similarity")
    ax1.set_title(f"Training curves — {prompt}")
    ax1.legend(fontsize=8)
    ax1.grid(alpha=0.2)

    ax2.plot(rewards, alpha=0.35, linewidth=0.8, label="Episode reward")
    if len(rewards) >= win:
        smoothed = np.convolve(rewards, np.ones(win) / win, mode="valid")
        ax2.plot(range(win - 1, len(rewards)), smoothed, linewidth=2, label=f"{win}-episode average")
    ax2.set_ylabel("Reward")
    ax2.set_xlabel("Episode")
    ax2.legend(fontsize=8)
    ax2.grid(alpha=0.2)

    plt.tight_layout()
    out_path = os.path.join(save_dir, "training_curve.png")
    plt.savefig(out_path, dpi=140)
    plt.close()
    print(f"[CURVE] Saved: {out_path}")


## 18. Training function


In [ ]:
def make_checkpoint(policy, vl, vl_model_id: str, best_sim: float, seed: Optional[int] = None, extra: Optional[dict] = None) -> dict:
    ckpt = {
        "policy_state_dict": policy.state_dict(),
        "vl_model_id": vl_model_id,
        "vl_embed_dim": vl.embed_dim,
        "state_dim": policy.state_dim,
        "action_dim": ACTION_DIM,
        "canvas_size": CANVAS_SIZE,
        "n_stroke_bundles": N_STROKE_BUNDLES,
        "strokes_per_step": STROKES_PER_STEP,
        "actor_log_std_init": ACTOR_LOG_STD_INIT,
        "best_sim": float(best_sim),
        "seed": None if seed is None else int(seed),
        "hyperparameters": {
            "reward_scale": REWARD_SCALE,
            "reward_clip": REWARD_CLIP,
            "stroke_size_penalty_w": STROKE_SIZE_PENALTY_W,
            "terminal_abs_w": TERMINAL_ABS_W,
            "div_bonus_w": DIV_BONUS_W,
            "entropy_start": ENTROPY_START,
            "entropy_end": ENTROPY_END,
            "lr": LR,
            "gamma": GAMMA,
            "lambda_gae": LAMBDA_GAE,
            "ppo_epochs": PPO_EPOCHS,
        },
    }
    if extra:
        ckpt.update(extra)
    return ckpt


def train(
    prompt: str,
    n_episodes: int,
    save_dir: str,
    device: torch.device,
    renderer_path: str = "auto",
    sketch_path: Optional[str] = None,
    vl_model_id: str = "google/siglip2-base-patch16-224",
    vl_dtype: torch.dtype = torch.float32,
    live_display: bool = True,
    display_every: int = 100,
    display_scale: int = 2,
    seed: Optional[int] = None,
):
    actual_seed = resolve_run_seed(seed)
    set_seed(actual_seed)
    os.makedirs(save_dir, exist_ok=True)
    renderer_path = resolve_renderer_path(renderer_path)

    vl = VisionLanguageEncoder(vl_model_id, device=device, dtype=vl_dtype)
    renderer = NeuralRenderer(renderer_path, device)

    with torch.no_grad():
        text_embed = vl.encode_text(prompt)
    sketch_embed = load_sketch_embed(sketch_path, vl, device)

    policy = ActorCritic(vl_embed_dim=vl.embed_dim).to(device)
    optimizer = optim.Adam(policy.parameters(), lr=LR)
    max_updates = max(1, math.ceil((n_episodes * N_STROKE_BUNDLES) / UPDATE_EVERY))
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_updates, eta_min=LR_MIN)
    buffer = RolloutBuffer()

    env = PaintEnv(text_embed, sketch_embed, vl, renderer, device)
    state, canvas = env.reset()

    sim_window, text_sim_window, cov_window = deque(maxlen=100), deque(maxlen=100), deque(maxlen=100)
    alpha_window, rew_window = deque(maxlen=100), deque(maxlen=100)
    all_sims, all_covs, all_alphas, all_rewards = [], [], [], []

    best_sim = -float("inf")
    best_canvas_np = None
    best_metrics = {}
    episodes_since_best = 0
    ep_num = 0
    update_count = 0
    last_update_loss = 0.0
    ep_reward = 0.0
    t_start = time.time()

    print(f"[INFO] prompt='{prompt}' | episodes={n_episodes} | device={device}")
    print(f"[INFO] seed={actual_seed} | save_dir={save_dir}")
    print(f"[INFO] STATE_DIM={policy.state_dim} | ACTION_DIM={ACTION_DIM}")
    print(f"[INFO] std init=exp({ACTOR_LOG_STD_INIT})={math.exp(ACTOR_LOG_STD_INIT):.4f}")
    print(f"[INFO] stroke penalty={STROKE_SIZE_PENALTY_W} | terminal={TERMINAL_ABS_W} | entropy={ENTROPY_START}->{ENTROPY_END}")
    print("-" * 72)

    while ep_num < n_episodes:
        progress = ep_num / max(n_episodes - 1, 1)
        entropy_coef = ENTROPY_START + progress * (ENTROPY_END - ENTROPY_START)

        canvas_before_cpu = env.get_canvas_snapshot()
        policy.eval()
        with torch.no_grad():
            action, log_p, val = policy.act(state.to(device), canvas.to(device), deterministic=False)

        next_state, next_canvas, reward, done, info = env.step(action)
        buffer.push(state, canvas_before_cpu, action, reward, log_p, val, done)
        ep_reward += reward
        state, canvas = next_state, next_canvas

        if len(buffer) >= UPDATE_EVERY:
            policy.train()
            with torch.no_grad():
                _, _, last_val = policy.act(state.to(device), canvas.to(device), deterministic=False)
            last_update_loss = ppo_update(policy, optimizer, buffer, last_val, device, entropy_coef)
            buffer.clear()
            scheduler.step()
            update_count += 1

        if done:
            ep_num += 1
            episodes_since_best += 1

            sim = info["vl_sim"]
            text_sim = info["text_sim"]
            cov = info["coverage"]
            alpha = info["mean_alpha"]
            canvas_np = canvas.detach().cpu().numpy().transpose(1, 2, 0)

            sim_window.append(sim)
            text_sim_window.append(text_sim)
            cov_window.append(cov)
            alpha_window.append(alpha)
            rew_window.append(ep_reward)
            all_sims.append(sim)
            all_covs.append(cov)
            all_alphas.append(alpha)
            all_rewards.append(ep_reward)

            current_metrics = {
                "episode": ep_num,
                "seed": int(actual_seed),
                "vl_sim": float(sim),
                "text_sim": float(text_sim),
                "coverage": float(cov),
                "alpha": float(alpha),
                "episode_reward": float(ep_reward),
                "entropy_coef": float(entropy_coef),
                "std_mean": float(policy.actor_log_std.exp().mean().detach().cpu()),
                "lr": float(optimizer.param_groups[0]["lr"]),
                "update_count": int(update_count),
                "last_update_loss": float(last_update_loss),
                "delta_sim_reward": float(info["delta_sim_reward"]),
                "stroke_penalty": float(info["stroke_penalty"]),
                "terminal_bonus": float(info["terminal_bonus"]),
            }

            if sim > best_sim:
                best_sim = sim
                best_canvas_np = canvas_np.copy()
                best_metrics = dict(current_metrics)
                best_metrics["best_sim"] = float(best_sim)
                _save_canvas(best_canvas_np, os.path.join(save_dir, "best.png"))
                torch.save(make_checkpoint(policy, vl, vl_model_id, best_sim, seed=actual_seed, extra={"best_metrics": best_metrics}), os.path.join(save_dir, "best_policy.pt"))
                save_json(best_metrics, os.path.join(save_dir, "best_metrics.json"))
                episodes_since_best = 0

            if SAVE_VISUAL_EVERY and ep_num % SAVE_VISUAL_EVERY == 0:
                _save_canvas(canvas_np, os.path.join(save_dir, f"visual_ep{ep_num:05d}.png"))

            if ep_num % display_every == 0 or ep_num == 1:
                elapsed = time.time() - t_start
                metrics = {
                    "episode": f"{ep_num}/{n_episodes}",
                    "vl_sim_avg100": float(np.mean(sim_window)),
                    "text_sim_avg100": float(np.mean(text_sim_window)),
                    "coverage_avg100": float(np.mean(cov_window)),
                    "alpha_avg100": float(np.mean(alpha_window)),
                    "reward_avg100": float(np.mean(rew_window)),
                    "best_sim": float(best_sim),
                    "entropy_coef": float(entropy_coef),
                    "std_mean": float(policy.actor_log_std.exp().mean().detach().cpu()),
                    "lr": float(optimizer.param_groups[0]["lr"]),
                    "updates": int(update_count),
                    "patience_counter": int(episodes_since_best),
                    "elapsed_seconds": int(elapsed),
                }
                print(
                    f"ep {ep_num:5d}/{n_episodes} | sim={metrics['vl_sim_avg100']:.4f} | "
                    f"cov={metrics['coverage_avg100']:.3f} | alpha={metrics['alpha_avg100']:.3f} | "
                    f"best={best_sim:.4f} | std={metrics['std_mean']:.3f} | ent={entropy_coef:.4f} | "
                    f"pat={episodes_since_best} | t={elapsed:.0f}s"
                )
                _save_canvas(canvas_np, os.path.join(save_dir, f"canvas_ep{ep_num:05d}.png"))
                if live_display:
                    fake_imshow(canvas_np, best_canvas_np, metrics, "Live SigLIP2 PPO drawing progress", os.path.join(save_dir, "live_progress.png"), display_scale)

            if episodes_since_best > PATIENCE:
                print(f"[STOP] No improvement for {PATIENCE} episodes. Early stopping.")
                break

            state, canvas = env.reset()
            ep_reward = 0.0

    print("-" * 72)
    print(f"[DONE] Best SigLIP reward similarity = {best_sim:.4f}")

    final_checkpoint = make_checkpoint(policy, vl, vl_model_id, best_sim, seed=actual_seed, extra={"best_metrics": best_metrics})
    torch.save(final_checkpoint, os.path.join(save_dir, "policy.pt"))
    save_json({"seed": int(actual_seed), "best_sim": float(best_sim), "best_metrics": best_metrics, "prompt": prompt}, os.path.join(save_dir, "run_summary.json"))

    if best_canvas_np is not None:
        Image.fromarray((np.clip(best_canvas_np, 0.0, 1.0) * 255).astype(np.uint8)).resize((256, 256), Image.NEAREST).save(os.path.join(save_dir, "best_256.png"))

    _plot_curves(all_sims, all_covs, all_alphas, all_rewards, prompt, save_dir)

    if best_canvas_np is not None and live_display:
        fake_imshow(best_canvas_np, best_canvas_np, {"best_sim": float(best_sim), "status": "training complete"}, "Final best canvas", os.path.join(save_dir, "final_best_preview.png"), display_scale)

    return best_canvas_np, best_sim


## 19. Evaluation function


In [ ]:
def evaluate(
    prompt: str,
    checkpoint_path: str,
    renderer_path: str,
    save_dir: str,
    device: torch.device,
    sketch_path: Optional[str] = None,
    vl_model_id: str = "google/siglip2-base-patch16-224",
    vl_dtype: torch.dtype = torch.float32,
    deterministic: bool = True,
    live_display: bool = True,
    display_scale: int = 2,
):
    os.makedirs(save_dir, exist_ok=True)
    renderer_path = resolve_renderer_path(renderer_path)

    ckpt_obj = _safe_torch_load(checkpoint_path, device)
    if not (isinstance(ckpt_obj, dict) and "policy_state_dict" in ckpt_obj):
        raise ValueError("This evaluator expects the cleaned checkpoint format with policy_state_dict.")

    policy_state = ckpt_obj["policy_state_dict"]
    ckpt_vl_model = ckpt_obj.get("vl_model_id", vl_model_id)
    if ckpt_vl_model != vl_model_id:
        print(f"[WARN] Using checkpoint VL model: {ckpt_vl_model}")
        vl_model_id = ckpt_vl_model

    if int(ckpt_obj.get("action_dim", ACTION_DIM)) != ACTION_DIM:
        raise ValueError("Checkpoint ACTION_DIM does not match current notebook constants. Use the same STROKES_PER_STEP/STROKE_PARAMS.")

    vl = VisionLanguageEncoder(vl_model_id, device=device, dtype=vl_dtype)
    if int(ckpt_obj.get("vl_embed_dim", vl.embed_dim)) != int(vl.embed_dim):
        raise ValueError("Checkpoint embedding dimension does not match the loaded SigLIP model.")

    with torch.no_grad():
        text_embed = vl.encode_text(prompt)
    sketch_embed = load_sketch_embed(sketch_path, vl, device)
    renderer = NeuralRenderer(renderer_path, device)
    policy = ActorCritic(vl_embed_dim=vl.embed_dim).to(device)
    policy.load_state_dict(policy_state)
    policy.eval()

    env = PaintEnv(text_embed, sketch_embed, vl, renderer, device)
    state, canvas = env.reset()
    frames = [canvas.cpu().numpy().transpose(1, 2, 0)]

    with torch.no_grad():
        for step in range(N_STROKE_BUNDLES):
            action, _, _ = policy.act(state.to(device), canvas.to(device), deterministic=deterministic)
            state, canvas, _, done, info = env.step(action)
            canvas_np = canvas.cpu().numpy().transpose(1, 2, 0)
            frames.append(canvas_np)
            print(
                f"bundle {step + 1:2d}/{N_STROKE_BUNDLES} | "
                f"vl_sim={info['vl_sim']:.4f} | text={info['text_sim']:.4f} | "
                f"cov={info['coverage']:.3f} | alpha={info['mean_alpha']:.3f}"
            )
            if live_display:
                fake_imshow(canvas_np, None, {"bundle": f"{step + 1}/{N_STROKE_BUNDLES}", "vl_sim": float(info["vl_sim"]), "coverage": float(info["coverage"]), "alpha": float(info["mean_alpha"])}, "Evaluation rollout", os.path.join(save_dir, "eval_live_preview.png"), display_scale)
            if done:
                break

    final_np = frames[-1]
    _save_canvas(final_np, os.path.join(save_dir, "eval_final.png"))

    gif_path = os.path.join(save_dir, "drawing.gif")
    pils = [Image.fromarray((np.clip(f, 0.0, 1.0) * 255).astype(np.uint8)).resize((256, 256), Image.NEAREST) for f in frames]
    pils[0].save(gif_path, save_all=True, append_images=pils[1:], duration=300, loop=0)
    print(f"[EVAL] GIF -> {gif_path}")
    return gif_path


## 20. Run training for selected seed mode

This cell trains the seeds selected by `RUN_MODE`:

- `RUN_MODE = "single"` trains only `SINGLE_SEED`.
- `RUN_MODE = "four_seed"` trains `[42, 123, 469, 1234]`.

Each seed is saved in a separate folder.

In [ ]:
RUN_RESULTS = []
LAST_RUN_SAVE_DIR = None
BEST_RUN_SAVE_DIR = None


def run_one_seed(seed_value: int):
    global LAST_RUN_SAVE_DIR, BEST_RUN_SAVE_DIR

    actual_seed = int(seed_value)
    run_save_dir = save_dir_for_seed(BASE_SAVE_DIR, actual_seed, USE_SEED_SUBFOLDERS)

    best_canvas_np, best_sim = train(
        prompt=PROMPT,
        n_episodes=N_EPISODES,
        save_dir=run_save_dir,
        device=DEVICE,
        renderer_path=RENDERER_PATH,
        sketch_path=SKETCH_PATH,
        vl_model_id=VL_MODEL_ID,
        vl_dtype=VL_DTYPE,
        live_display=LIVE_DISPLAY,
        display_every=DISPLAY_EVERY,
        display_scale=DISPLAY_SCALE,
        seed=actual_seed,
    )

    result = {
        "seed": int(actual_seed),
        "best_sim": float(best_sim),
        "save_dir": run_save_dir,
        "prompt": PROMPT,
    }
    RUN_RESULTS.append(result)
    LAST_RUN_SAVE_DIR = run_save_dir
    BEST_RUN_SAVE_DIR = max(RUN_RESULTS, key=lambda r: r["best_sim"])["save_dir"]

    os.makedirs(BASE_SAVE_DIR, exist_ok=True)
    save_json(RUN_RESULTS, os.path.join(BASE_SAVE_DIR, "seed_results.json"))
    print("Run result:", result)
    print("Current best run folder:", BEST_RUN_SAVE_DIR)

    # Free memory before the next seed, especially in Colab/Kaggle notebooks.
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return best_canvas_np, best_sim


print(f"RUN_MODE={RUN_MODE} | Running seeds: {RUN_SEEDS}")
for seed_value in RUN_SEEDS:
    best_canvas_np, best_sim = run_one_seed(seed_value)

print("\nAll seed results:")
for result in RUN_RESULTS:
    print(result)

print("Best run folder:", BEST_RUN_SAVE_DIR)
print("Seed results saved to:", os.path.join(BASE_SAVE_DIR, "seed_results.json"))


## 21. Deterministic evaluation for every seed

This cell runs evaluation for every seed folder and saves `drawing.gif` in each folder.

In [ ]:
import os
import glob

BASE = globals().get("BASE_SAVE_DIR", None)
if BASE is None:
    BASE = "/kaggle/working/outputs_siglip" if os.path.exists("/kaggle/working") else "/content/outputs_siglip"

if EVAL_SAVE_DIR_OVERRIDE is not None:
    eval_dirs = [EVAL_SAVE_DIR_OVERRIDE]
else:
    expected_dirs = [save_dir_for_seed(BASE, int(seed), USE_SEED_SUBFOLDERS) for seed in RUN_SEEDS]
    eval_dirs = [d for d in expected_dirs if os.path.isdir(d)]

    # Fallback if the folder naming was changed manually.
    if not eval_dirs:
        eval_dirs = sorted(d for d in glob.glob(os.path.join(BASE, "seed_*")) if os.path.isdir(d))

if not eval_dirs:
    raise FileNotFoundError(f"No seed folders found for evaluation in: {BASE}")

print("Evaluation folders:")
for d in eval_dirs:
    print(" -", d)

EVALUATION_RESULTS = []

for EVAL_SAVE_DIR in eval_dirs:
    print("\n" + "=" * 80)
    print("Evaluating folder:", EVAL_SAVE_DIR)

    checkpoint_path = os.path.join(EVAL_SAVE_DIR, "best_policy.pt")
    if not os.path.isfile(checkpoint_path):
        checkpoint_path = os.path.join(EVAL_SAVE_DIR, "policy.pt")

    if os.path.isfile(checkpoint_path):
        print("Evaluating checkpoint:", checkpoint_path)
        gif_path = evaluate(
            prompt=PROMPT,
            checkpoint_path=checkpoint_path,
            renderer_path=RENDERER_PATH,
            save_dir=EVAL_SAVE_DIR,
            device=DEVICE,
            sketch_path=SKETCH_PATH,
            vl_model_id=VL_MODEL_ID,
            vl_dtype=VL_DTYPE,
            deterministic=True,
            live_display=LIVE_DISPLAY,
            display_scale=DISPLAY_SCALE,
        )
        EVALUATION_RESULTS.append({"save_dir": EVAL_SAVE_DIR, "checkpoint": checkpoint_path, "gif": gif_path})
    else:
        print("No checkpoint found in:", EVAL_SAVE_DIR)
        EVALUATION_RESULTS.append({"save_dir": EVAL_SAVE_DIR, "checkpoint": None, "gif": None})

save_json(EVALUATION_RESULTS, os.path.join(BASE, "evaluation_results.json"))
print("\nEvaluation results saved to:", os.path.join(BASE, "evaluation_results.json"))


## 22. Inspect saved output files

In [ ]:
BASE = globals().get("BASE_SAVE_DIR", "/content/outputs_siglip")
print("Base output folder:", BASE)

seed_dirs = [save_dir_for_seed(BASE, int(seed), USE_SEED_SUBFOLDERS) for seed in RUN_SEEDS]
seed_dirs = [d for d in seed_dirs if os.path.isdir(d)]

for seed_dir in seed_dirs:
    print("\n" + "=" * 80)
    print("Inspecting:", seed_dir)
    for name in ["best.png", "best_256.png", "drawing.gif", "training_curve.png", "best_metrics.json", "run_summary.json"]:
        path = os.path.join(seed_dir, name)
        print(("[OK] " if os.path.isfile(path) else "[MISS] ") + path)

seed_results_path = os.path.join(BASE, "seed_results.json")
if os.path.isfile(seed_results_path):
    print("\nSeed results:")
    with open(seed_results_path, "r", encoding="utf-8") as f:
        print(f.read())


## 23. Verify training curves

The training function now saves one `training_curve.png` per seed containing only similarity and reward.

In [ ]:
BASE = globals().get("BASE_SAVE_DIR", "/content/outputs_siglip")
seed_dirs = [save_dir_for_seed(BASE, int(seed), USE_SEED_SUBFOLDERS) for seed in RUN_SEEDS]
seed_dirs = [d for d in seed_dirs if os.path.isdir(d)]

for seed_dir in seed_dirs:
    curve_path = os.path.join(seed_dir, "training_curve.png")
    if os.path.isfile(curve_path):
        print(f"[OK] {curve_path}")
    else:
        print(f"[WARN] Missing training curve: {curve_path}")


In [ ]:
import os
import glob
import zipfile
from IPython.display import FileLink, display

BASE = globals().get("BASE_SAVE_DIR", "/content/outputs_siglip")

FILES_TO_ZIP = [
    "drawing.gif",
    "training_curve.png",
    "best.png",
    "best_256.png",
    "best_metrics.json",
    "run_summary.json",
]

seed_dirs = [save_dir_for_seed(BASE, int(seed), USE_SEED_SUBFOLDERS) for seed in RUN_SEEDS]
seed_dirs = [d for d in seed_dirs if os.path.isdir(d)]

# Fallback if folders were created manually.
if not seed_dirs:
    seed_dirs = sorted(d for d in glob.glob(os.path.join(BASE, "seed_*")) if os.path.isdir(d))

if not seed_dirs:
    raise FileNotFoundError(f"No seed folders found in: {BASE}")


def find_file(seed_dir, filename):
    """Find a file directly or recursively inside a seed folder."""
    direct_path = os.path.join(seed_dir, filename)
    if os.path.isfile(direct_path):
        return direct_path

    matches = glob.glob(os.path.join(seed_dir, "**", filename), recursive=True)
    matches = [m for m in matches if os.path.isfile(m)]

    if matches:
        return max(matches, key=os.path.getmtime)

    return None

zip_path = os.path.join(BASE, f"{RUN_MODE}_seed_results.zip")

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    # Add global summary files at the root of the archive.
    for root_file in ["seed_results.json", "evaluation_results.json"]:
        root_path = os.path.join(BASE, root_file)
        if os.path.isfile(root_path):
            zf.write(root_path, arcname=root_file)

    for seed_dir in seed_dirs:
        seed_name = os.path.basename(seed_dir)
        existing_files = []
        missing_files = []

        for fname in FILES_TO_ZIP:
            fpath = find_file(seed_dir, fname)
            if fpath is None:
                missing_files.append(fname)
            else:
                existing_files.append((fname, fpath))

        if not existing_files:
            print(f"[SKIP] No target files found for {seed_name}")
            continue

        for fname, fpath in existing_files:
            zf.write(fpath, arcname=os.path.join(seed_name, fname))

        print(f"[ADDED] {seed_name}")
        if missing_files:
            print(f"[WARN] {seed_name} missing: {missing_files}")

print("\n[ZIP] Created one combined zip file:")
print(zip_path)

display(FileLink(zip_path))
